In [1]:
# DEPENDENCIES

import sys
sys.path.insert(0, '..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

In [2]:
# SET-UP
SUBJECTS = helper_functions.alice_get_subjects()
ENVELOPES_LOG = eelbrain.load.unpickle(ALICE_PROCESSED_PREDICTOR_DIR / f'~processed_envelopes-log.pickle')
STIMULI = [str(i) for i in range(1, 13)]
DURATIONS = helper_functions.alice_get_durations(ENVELOPES_LOG, STIMULI)

LOW_FREQUENCY = 0.5
HIGH_FREQUENCY = 20

# Define models
trf_types = [
    # Single predictors
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD, False),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.SINGLE, MODEL_TYPE.BACKWARD, False),

    # Multiple predictors
    #([PREDICTOR_TYPE.ENVELOPE, PREDICTOR_TYPE.ENVELOPE_ONSET], ATTENTION_TYPE.SINGLE, MODEL_TYPE.FORWARD, False),
]

In [3]:
# LOAD EEG DATA DOWNSAMPLED TO 64 HZ

# Iterate over the EEG of every subject
for subject in SUBJECTS:
    # Extract the raw files
    raw = mne.io.read_raw(ALICE_EEG_DIR / subject / f'{subject}_alice-raw.fif', preload=True)
    # Filter the raw data to the desired band
    raw.filter(LOW_FREQUENCY, HIGH_FREQUENCY, n_jobs=1)
    # Interpolate bad channels
    raw.interpolate_bads() # <-- No bad filters so commented out

    # Load the events embeded in the raw file
    events = eelbrain.load.fiff.events(raw)

    # Not all subjects have all trials; determine which stimuli are present
    trial_indexes = [STIMULI.index(stimulus) for stimulus in events['event']]
    # Extract the EEG data segments corresponding to the stimuli
    trial_durations = [DURATIONS[i] for i in trial_indexes]
    # Load the EEG with corresponding epochs
    eeg = eelbrain.load.fiff.variable_length_epochs(events, 
                                                    -0.100, 
                                                    trial_durations) # <-- Separates by events, adds 100ms pre-stimulus, no decimation anymore
    # Since trials are of unequal length, we will concatenate them for the TRF estimation.
    eeg_concatenated = eelbrain.concatenate(eeg)
    #print(eeg_concatenated)

    # Resample to 64 Hz <-- Most important change for this code snippet
    eeg_resampled = eelbrain.resample(eeg_concatenated, 64)
    #print(eeg_resampled.time.tstep)

    # Save pickle
    eelbrain.save.pickle(eeg_resampled, ALICE_EEG_DIR / subject / f'{subject} 64hz_concatenated_eeg.pickle')

print('\nProcess Completed! \n:)')


/var/folders/y0/p95jrf3s35b2tqj_rnj24xjr0000gn/T/ipykernel_18223/3101932940.py:10: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw.interpolate_bads() # <-- No bad filters so commented out



Process Completed! 
:)


In [4]:
# LOAD PREDICTORS DOWNSAMPLED TO 64 HZ

# Log
envelopes_log = [eelbrain.load.unpickle(ALICE_PREDICTOR_DIR / f'{stimulus}~envelope-log.pickle') for stimulus in STIMULI]
envelopes_log = [eelbrain.resample(x, 64) for x in envelopes_log]  # ← same as EEG
envelopes_log = [eelbrain.pad(x, tstart=-0.100, tstop=x.time.tstop+1, name='envelope_log') for x in envelopes_log]
envelopes_log = [eelbrain.filter_data(x, LOW_FREQUENCY, HIGH_FREQUENCY) for x in envelopes_log]

# Onset
envelopes_onset = [eelbrain.load.unpickle(ALICE_PREDICTOR_DIR / f'{stimulus}~envelope-onset.pickle') for stimulus in STIMULI]
envelopes_onset = [eelbrain.resample(x, 64) for x in envelopes_onset]  # ← same as EEG
envelopes_onset = [eelbrain.pad(x, tstart=-0.100, tstop=x.time.tstop+1, name='envelope_onset') for x in envelopes_onset]
envelopes_onset = [eelbrain.filter_data(x, LOW_FREQUENCY, HIGH_FREQUENCY) for x in envelopes_onset]

# Concatenate for the different trials
for subject in SUBJECTS:
    raw = mne.io.read_raw(ALICE_EEG_DIR / subject / f'{subject}_alice-raw.fif', preload=True)
    events = eelbrain.load.fiff.events(raw)
    trial_indexes = [STIMULI.index(stimulus) for stimulus in events['event']]
    trial_durations = [DURATIONS[i] for i in trial_indexes]

    # Get per-trial EEG lengths to match predictors against
    eeg_epochs = eelbrain.load.fiff.variable_length_epochs(events, -0.100, trial_durations)
    eeg_epochs_resampled = [eelbrain.resample(epoch, 64) for epoch in eeg_epochs]


    # 64 hz sample causes length mismatch in events ending exactly in numbers divisible by 64, adding or removing +- 1 sample
    # We introduce correction function to match the samples directly
    def match_length(pred, eeg_epoch):
        n_diff = pred.time.nsamples - eeg_epoch.time.nsamples
        if n_diff > 0:
            return pred.sub(time=(pred.time.tmin, pred.time.tmin + eeg_epoch.time.nsamples / 64))
        return pred

    envelopes_log_concat = eelbrain.concatenate([match_length(envelopes_log[idx], epoch) 
                                                  for idx, epoch in zip(trial_indexes, eeg_epochs_resampled)])
    envelopes_onset_concat = eelbrain.concatenate([match_length(envelopes_onset[idx], epoch) 
                                                    for idx, epoch in zip(trial_indexes, eeg_epochs_resampled)])
    
    # ————————————————————————————————————————————————————————————————————————————————————————————————
    # Save pickles

    eelbrain.save.pickle(envelopes_log_concat, 
                            ALICE_PROCESSED_PREDICTOR_DIR / f'{subject} 64hz-processed-envelope_log.pickle') # <-- This is referred to as gammatone-1 in document
    eelbrain.save.pickle(envelopes_onset_concat, 
                            ALICE_PROCESSED_PREDICTOR_DIR / f'{subject} 64hz-processed-envelope_log_onset.pickle') # <-- This is referred to as gammatone-on-1 in document
    
print('\nProcess Completed! \n:)')


Process Completed! 
:)


In [ ]:
# ESTIMATE ALIGNED BACKWARD MODEL TRFS WITH 64HZ DATA

# Iterative loop to estimate every decoder TRF per-subject
for subject in SUBJECTS:
    # ————————————————————————————————————————————————————————————————————————————————————————————————
    # Load eeg
    eeg = eelbrain.load.unpickle(ALICE_EEG_DIR / subject / f'{subject} 64hz_concatenated_eeg.pickle')
    print('——————————————————————————————')
    print(f'{subject} eeg extracted')

    for type in trf_types:  
        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Get the model name
        p, a, w, pad = type
        model = helper_functions.get_trf_model_name(DATASET_TYPE.ALICE, *type)

        if isinstance(p, PREDICTOR_TYPE):
            predictor_type = helper_functions.map_predictor_name(p, DATASET_TYPE.ALICE)
            predictors = eelbrain.load.unpickle(ALICE_PROCESSED_PREDICTOR_DIR / f'{subject} 64hz-processed-{predictor_type}.pickle')
        else:
            predictors = []
            for _ in p:
                predictor_type = helper_functions.map_predictor_name(_, DATASET_TYPE.ALICE) 
                predictors.append(eelbrain.load.unpickle(ALICE_PROCESSED_PREDICTOR_DIR / f'{subject} 64hz-processed-{predictor_type}.pickle'))

        trf = eelbrain.boosting(predictors, eeg, -1.000, 0.100, 
                                error='l1', basis=0.050, partitions=5, test=1, selective_stopping=True)

        # ————————————————————————————————————————————————————————————————————————————————————————————————
        # Save in directory
        eelbrain.save.pickle(trf, ALICE_TRF_DIR / subject / f'{subject} 64hz-{model}.pickle')
        print(f'TRF for {model} complete')
    
print('\nProcess Completed! \n:)')

——————————————————————————————
S01 eeg extracted
TRF for decoder-envelope_log complete
